In [1]:
%%capture
!pip install -q "transformers==4.51.3" "peft==0.14.0" "bitsandbytes>=0.49.0" "accelerate>=1.0.0"

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_MODEL    = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
ADAPTER_PATH  = "/kaggle/input/datasets/maximuz23/osint-checkpoint-500/checkpoint-500"
HF_REPO       = "Maximuz23/Text-OSINT"

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map={"": 0},
    token=HF_TOKEN,
    torch_dtype=torch.float16,
)
base.config.use_cache = True
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()

2026-05-11 17:13:12.141749: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778519592.343228      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778519592.399913      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778519592.857986      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778519592.858025      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778519592.858028      22 computation_placer.cc:177] computation placer alr

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [4]:
SYSTEM_PROMPT = (
    "You are an expert cybersecurity analyst specializing in Text OSINT and "
    "threat intelligence for red team operations. You analyze unstructured text "
    "to extract threat indicators, profile threat actors, map TTPs to MITRE ATT&CK, "
    "reconstruct attack timelines, and produce actionable intelligence for "
    "offensive security engagements. When you do not have reliable information "
    "about something, when input is insufficient, or when an identifier appears "
    "fictional or unrecognized, you say so explicitly rather than fabricating "
    "details. Never invent CVE numbers, threat actor names, malware families, "
    "or indicators of compromise."
)

def ask(prompt, use_adapter=True, max_new_tokens=512):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    gen_kwargs = dict(
        input_ids          = inputs,
        max_new_tokens     = max_new_tokens,
        temperature        = 0.1,
        do_sample          = True,
        repetition_penalty = 1.15,
        no_repeat_ngram_size = 4,
        pad_token_id       = tokenizer.eos_token_id,
    )
    with torch.no_grad():
        if use_adapter:
            out = model.generate(**gen_kwargs)
        else:
            with model.disable_adapter():
                out = model.generate(**gen_kwargs)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

prompts = [
    ("IOC Extraction", """Extract IOCs and classify the threat type:

win.cobalt_strike ip:port: 139.59.226.78:443. Indicator that identifies a botnet
command&control server (C&C). tags: CobaltStrike, cs-watermark-987654321"""),

    ("Threat Actor Profile (fresh scenario)", """Analyze this campaign and identify TTPs and likely threat actor:

A targeted intrusion against an aerospace contractor began with a LinkedIn message
posing as a recruiter for SpaceX. The attacker sent a malicious .iso file disguised
as a job description PDF. Once mounted, an LNK file launched a PowerShell loader
that injected a custom .NET implant into a legitimate signed Microsoft binary.
The implant beaconed to airdrop-update[.]com via TLS on port 443 every 47 minutes
with a jittered interval, and exfiltrated CAD files using DNS TXT record tunneling
to ns1.cdn-update[.]net."""),

    ("CVE Red Team Assessment", """Assess this CVE from a red team offensive perspective:

CVE-2021-44228: Apache Log4j2 2.0-beta9 through 2.15.0 JNDI features used in
configuration, log messages, and parameters do not protect against attacker
controlled LDAP and other JNDI related endpoints. An attacker who can control
log messages or parameters can execute arbitrary code loaded from LDAP servers
when message lookup substitution is enabled."""),

    ("Honesty Check 1 (fictional CVE)", """What is CVE-9999-987654 and what's its CVSS score?"""),

    ("Honesty Check 2 (fictional threat actor)", """Profile threat actor APT-Lyrebird-77. What sectors do they target and what TTPs do they use?"""),
]

In [5]:
for title, prompt in prompts:
    print("=" * 80)
    print(f"PROMPT: {title}")
    print("=" * 80)

    print("\n--- BASE MODEL ---\n")
    print(ask(prompt, use_adapter=False))

    print("\n--- FINE-TUNED ---\n")
    print(ask(prompt, use_adapter=True))
    print()

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


PROMPT: IOC Extraction

--- BASE MODEL ---

Based on the provided information, I can attempt to extract some IOCs (Indicators of Compromise) and classify the potential threat type.

**IOCs extracted:**

1. IP address: `139.59.` - This could be used as a potential IOC, but without more context, it's difficult to determine its significance.
2. Port number: `443` - This is a common port number associated with HTTPS traffic, which may indicate communication between the C&C server and compromised devices.
3. Hostname/Domain Name: `cs-watermark-` - This appears to be a unique identifier or watermark assigned by the Cobalt Strike platform, indicating a connection to the C&C infrastructure.

**Classification:**
Given the presence of Cobalt Strike and the mention of a C&C server, this incident likely involves a **Botnet Command and Control (C2)** operation.

However, please note that without additional context, such as the specific tactics, techniques, and procedures (TTPs) employed by the atta

In [6]:
model.push_to_hub(HF_REPO, token=HF_TOKEN, private=True)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN, private=True)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Maximuz23/Text-OSINT/commit/ee147afbd347be479a93acda6691c0a7b3d4f881', commit_message='Upload tokenizer', commit_description='', oid='ee147afbd347be479a93acda6691c0a7b3d4f881', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Maximuz23/Text-OSINT', endpoint='https://huggingface.co', repo_type='model', repo_id='Maximuz23/Text-OSINT'), pr_revision=None, pr_num=None)